In [1]:
!pip install -U "typing_extensions>=4.12" vllm transformers datasets accelerate sentencepiece

  Using cached transformers-5.3.0-py3-none-any.whl.metadata (32 kB)

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip


In [5]:
import json
import subprocess

MODEL_ALIASES = [
    "qwen2.5-0.5b",
    "qwen2.5-1.5b",
    "qwen2.5-3b",
    "smollm2-1.7b",
    "granite-3.3-2b",
    "nemotron-mini-4b",
    "phi-4-mini",
]

results = []

for alias in MODEL_ALIASES:
    print(f"\nRunning {alias} ...")
    proc = subprocess.run(
        [
            "python",
            "run_one_vllm_benchmark.py",
            "--model-alias", alias,
            "--sample-size", "500",
            "--seed", "42",
            "--max-new-tokens", "512",
            "--gpu-memory-utilization", "0.8",
            "--max-model-len", "4608",
        ],
        capture_output=True,
        text=True,
    )

    if proc.returncode != 0:
        print(proc.stderr)
        results.append({
            "model": alias,
            "error": proc.stderr.strip() or proc.stdout.strip() or f"exit code {proc.returncode}",
        })
        continue

    lines = [line.strip() for line in proc.stdout.splitlines() if line.strip()]
    try:
        result = json.loads(lines[-1])
        results.append(result)
        print(result)
    except Exception:
        results.append({
            "model": alias,
            "error": proc.stdout[-2000:],
        })

results


Running qwen2.5-0.5b ...
{'model': 'qwen2.5-0.5b', 'hf_id': 'Qwen/Qwen2.5-0.5B-Instruct', 'eval_time_s': 1998.021, 'tp': 4275, 'fp': 3931, 'fn': 2100, 'precision': 0.521, 'recall': 0.6706, 'f1': 0.5864, 'samples_processed': 500, 'prompt_tokens': 874013, 'output_tokens': 186659, 'generation_time_s': 1994.97, 'avg_latency_per_sample_s': 3.99, 'avg_output_tokens_per_sample': 373.318, 'output_tokens_per_s': 93.565}

Running qwen2.5-1.5b ...
{'model': 'qwen2.5-1.5b', 'hf_id': 'Qwen/Qwen2.5-1.5B-Instruct', 'eval_time_s': 1363.297, 'tp': 2611, 'fp': 2086, 'fn': 1542, 'precision': 0.5559, 'recall': 0.6287, 'f1': 0.5901, 'samples_processed': 500, 'prompt_tokens': 874013, 'output_tokens': 110387, 'generation_time_s': 1360.168, 'avg_latency_per_sample_s': 2.72, 'avg_output_tokens_per_sample': 220.774, 'output_tokens_per_s': 81.157}

Running qwen2.5-3b ...
{'model': 'qwen2.5-3b', 'hf_id': 'Qwen/Qwen2.5-3B-Instruct', 'eval_time_s': 1138.201, 'tp': 2049, 'fp': 721, 'fn': 1847, 'precision': 0.7397, 

[{'model': 'qwen2.5-0.5b',
  'hf_id': 'Qwen/Qwen2.5-0.5B-Instruct',
  'eval_time_s': 1998.021,
  'tp': 4275,
  'fp': 3931,
  'fn': 2100,
  'precision': 0.521,
  'recall': 0.6706,
  'f1': 0.5864,
  'samples_processed': 500,
  'prompt_tokens': 874013,
  'output_tokens': 186659,
  'generation_time_s': 1994.97,
  'avg_latency_per_sample_s': 3.99,
  'avg_output_tokens_per_sample': 373.318,
  'output_tokens_per_s': 93.565},
 {'model': 'qwen2.5-1.5b',
  'hf_id': 'Qwen/Qwen2.5-1.5B-Instruct',
  'eval_time_s': 1363.297,
  'tp': 2611,
  'fp': 2086,
  'fn': 1542,
  'precision': 0.5559,
  'recall': 0.6287,
  'f1': 0.5901,
  'samples_processed': 500,
  'prompt_tokens': 874013,
  'output_tokens': 110387,
  'generation_time_s': 1360.168,
  'avg_latency_per_sample_s': 2.72,
  'avg_output_tokens_per_sample': 220.774,
  'output_tokens_per_s': 81.157},
 {'model': 'qwen2.5-3b',
  'hf_id': 'Qwen/Qwen2.5-3B-Instruct',
  'eval_time_s': 1138.201,
  'tp': 2049,
  'fp': 721,
  'fn': 1847,
  'precision': 0.7397,

In [6]:
successful_results = [r for r in results if "error" not in r]
failed_results = [r for r in results if "error" in r]

successful_results = sorted(
    successful_results,
    key=lambda r: (r["f1"], r["output_tokens_per_s"]),
    reverse=True
)

print("\n===== BENCHMARK SUMMARY =====")
for r in successful_results:
    print(
        f"{r['model']:18s} | "
        f"F1={r['f1']:.4f} | "
        f"P={r['precision']:.4f} | "
        f"R={r['recall']:.4f} | "
        f"tok/s={r['output_tokens_per_s']:.2f} | "
        f"lat/sample={r['avg_latency_per_sample_s']:.2f}s | "
        f"eval_time={r['eval_time_s']:.1f}s"
    )

if failed_results:
    print("\n===== FAILED MODELS =====")
    for r in failed_results:
        print(f"{r['model']:18s} | {r['error'][:300]}")


===== BENCHMARK SUMMARY =====
qwen2.5-3b         | F1=0.6148 | P=0.7397 | R=0.5259 | tok/s=61.84 | lat/sample=2.27s | eval_time=1138.2s
granite-3.3-2b     | F1=0.6096 | P=0.6014 | R=0.6180 | tok/s=51.48 | lat/sample=3.66s | eval_time=1832.4s
qwen2.5-1.5b       | F1=0.5901 | P=0.5559 | R=0.6287 | tok/s=81.16 | lat/sample=2.72s | eval_time=1363.3s
qwen2.5-0.5b       | F1=0.5864 | P=0.5210 | R=0.6706 | tok/s=93.56 | lat/sample=3.99s | eval_time=1998.0s
phi-4-mini         | F1=0.4782 | P=0.4490 | R=0.5115 | tok/s=48.33 | lat/sample=4.60s | eval_time=2304.0s
smollm2-1.7b       | F1=0.4059 | P=0.3348 | R=0.5155 | tok/s=105.88 | lat/sample=3.45s | eval_time=1726.5s

===== FAILED MODELS =====
nemotron-mini-4b   | Traceback (most recent call last):
  File "/workspace/run_one_vllm_benchmark.py", line 259, in <module>
    main()
  File "/workspace/run_one_vllm_benchmark.py", line 165, in main
    llm = LLM(
          ^^^^
  File "/usr/local/lib/python3.11/dist-packages/vllm/entrypoints/llm.py", 

In [7]:
import json
import csv
from pathlib import Path

# Output folder
out_dir = Path("benchmark_outputs")
out_dir.mkdir(exist_ok=True)

# Split results
successful_results = [r for r in results if "error" not in r]
failed_results = [r for r in results if "error" in r]

# Sort successful results by F1, then speed
successful_results = sorted(
    successful_results,
    key=lambda r: (r["f1"], r["output_tokens_per_s"]),
    reverse=True
)

# 1) Save full raw results as JSON
json_path = out_dir / "benchmark_results.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(
        {
            "successful_results": successful_results,
            "failed_results": failed_results,
            "all_results": results,
        },
        f,
        indent=2,
        ensure_ascii=False,
    )

# 2) Save successful results as CSV
csv_path = out_dir / "benchmark_results.csv"
csv_fields = [
    "model",
    "f1",
    "precision",
    "recall",
    "output_tokens_per_s",
    "avg_latency_per_sample_s",
    "eval_time_s",
]

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=csv_fields)
    writer.writeheader()
    for r in successful_results:
        writer.writerow({k: r.get(k, "") for k in csv_fields})

# 3) Save human-readable summary
txt_path = out_dir / "benchmark_summary.txt"
with open(txt_path, "w", encoding="utf-8") as f:
    f.write("===== BENCHMARK SUMMARY =====\n")
    for r in successful_results:
        f.write(
            f"{r['model']:18s} | "
            f"F1={r['f1']:.4f} | "
            f"P={r['precision']:.4f} | "
            f"R={r['recall']:.4f} | "
            f"tok/s={r['output_tokens_per_s']:.2f} | "
            f"lat/sample={r['avg_latency_per_sample_s']:.2f}s | "
            f"eval_time={r['eval_time_s']:.1f}s\n"
        )

    if failed_results:
        f.write("\n===== FAILED MODELS =====\n")
        for r in failed_results:
            err = r.get("error", "")[:300].replace("\n", " ")
            f.write(f"{r['model']:18s} | {err}\n")

print(f"Saved JSON to: {json_path}")
print(f"Saved CSV to:  {csv_path}")
print(f"Saved TXT to:  {txt_path}")

Saved JSON to: benchmark_outputs/benchmark_results.json
Saved CSV to:  benchmark_outputs/benchmark_results.csv
Saved TXT to:  benchmark_outputs/benchmark_summary.txt
